In [184]:
import os
import re
import pandas as pd
import numpy as np

### University Data Cleaning step

In [185]:
def parse_intl_eligibility(val):
    if pd.isna(val):
        return 0
    clean_val = str(val).strip().upper()
    if "NOT ELIGIBLE" in clean_val or "DOMESTIC ONLY" in clean_val:
        return 0
    if "ELIGIBLE" in clean_val or "OPEN" in clean_val:
        return 1
    return 0


In [186]:
def extract_absolute_minimum_percentage(val, default=0.0):
    """
    Uses RegEx to pull numeric values from text. If a range or multiple numbers 
    are encountered, it strictly isolates and chooses the absolute minimum value.
    E.g., "55–65%" -> 55.0, "94-100" -> 94.0, "80% or 75% for State Boards" -> 75.0
    """
    if pd.isna(val):
        return default
    
    clean_val = str(val).replace(",", "").strip()
    
    # Find all standalone integers or decimals in the text line
    all_numbers = re.findall(r'(\d+(?:\.\d+)?)\s*%', clean_val)
    if not all_numbers:
        return default
        
    # Cast array items to floats and isolate the absolute minimum
    float_numbers = [float(n) for n in all_numbers if 0.0 < float(n) <= 100.0]
    if float_numbers:
        return min(float_numbers)
        
    return default

In [187]:
def parse_bachelor_percentage_regex(val):
    """
    Extracts the minimum bachelor marks threshold from the min_bachelor_requirement column.
    Safely parses percentages and structural NZ/Indian GPA ratios, prioritizing the minimum boundary.
    """
    if pd.isna(val):
        return 55.0  
    
    clean_val = str(val).strip().lower()
    
    # 1. Parse standard percentage expressions or ranges (choosing the minimum)
    pct_matches = re.findall(r"(\d+)\s*%", clean_val)
    if pct_matches:
        return min([float(p) for p in pct_matches])
        
    # 2. Parse structural GPA scales on a 9-point scale (GPE/GPA mapping)
    gpa_9_matches = re.findall(r"(?:gpa|gpe)\s*(\d+(?:\.\d+)?)\s*/\s*9", clean_val)
    if gpa_9_matches:
        min_gpa = min([float(g) for g in gpa_9_matches])
        return 50.0 + (min_gpa * 2.5)  # E.g., 4.0/9 -> 60.0%
        
    # 3. Parse structural GPA scales on a 4-point scale
    gpa_4_matches = re.findall(r"(?:gpa|gpe)\s*(\d+(?:\.\d+)?)\s*/\s*4", clean_val)
    if gpa_4_matches:
        min_gpa = min([float(g) for g in gpa_4_matches])
        return 50.0 + (min_gpa * 10.0)
        
    # 4. Semantic textual fallback mapping scales
    if "b+ average" in clean_val or "gpa 6.0+" in clean_val or "gpe 5.0" in clean_val:
        return 65.0
    if any(w in clean_val for w in ["b average", "second-class", "b- average", "first class"]):
        return 60.0
        
    # 5. Extract all numbers and locate the absolute minimum boundary float
    return extract_absolute_minimum_percentage(clean_val, default=55.0)



In [188]:
def parse_work_experience(req_text):
    if pd.isna(req_text):
        return 0, 0

    clean_text = str(req_text).strip().lower()
    if clean_text in ["none", "no", "none specified"] or "not formally mandated" in clean_text:
        return 0, 0

    is_mandatory = int(
        any(w in clean_text for w in ["required", "must be", "mandatory", "yes"])
    )

    year_match = re.search(r"(\d+)\s*\+?\s*-?\s*(?:years?|yrs?)\b", clean_text)
    if year_match:
        return is_mandatory, int(year_match.group(1)) * 12

    return is_mandatory, 0

In [189]:
def extract_bachelor_years_regex(combined_context):
    """
    Deterministic rule-based RegEx scanner replacing the LLM.
    Scans the unified text block for strict flags indicating a 4-year degree is mandatory.
    """
    clean_text = str(combined_context).lower()
    
    # Rules targeting strict 4-year constraints
    four_year_indicators = [
        r"four-year", 
        r"4-year", 
        r"be/btech", 
        r"b\.tech", 
        r"b\.e\.",
        r"honours degree", 
        r"barch", 
        r"b\.arch",
        r"insufficient and would need topping up",
        r"three-year.*insufficient"
    ]
    
    # Exceptions that explicitly state an Indian 3-year degree is okay
    three_year_allowed_indicators = [
        r"3-year.*accepted", 
        r"3-year or 4-year.*accepted", 
        r"3-year.*is fine", 
        r"open to non-accounting graduates",
        r"conversion master"
    ]
    
    # Evaluate 3-year allowance overrides first
    for pattern in three_year_allowed_indicators:
        if re.search(pattern, clean_text):
            return 3
            
    # Evaluate strict 4-year boundaries
    for pattern in four_year_indicators:
        if re.search(pattern, clean_text):
            return 4
            
    return 3  # Standard system fallback


In [190]:
def extract_work_exp_substitution_regex(combined_context):
    """
    Deterministic rule-based RegEx scanner replacing the LLM.
    Identifies if alternative work experience context allows a degree requirement waiver.
    """
    clean_text = str(combined_context).lower()
    
    substitution_patterns = [
        r"experience may substitute",
        r"experience can substitute",
        r"experience in lieu",
        r"professional practice may substitute",
        r"experience.*accepted as an alternative entry",
        r"experience can substitute for standard grades",
        r"experience.*substitute for missing subject"
    ]
    
    for pattern in substitution_patterns:
        if re.search(pattern, clean_text):
            return 1
            
    return 0


In [191]:
def dynamically_match_field_prereq(row):
    """
    Dynamically checks if the academic field mentioned in the 'field' column 
    is explicitly requested inside the 'academic_prerequisites' for that row.
    Handles semantic overlaps and splits multiple domain descriptors recursively.
    """
    if pd.isna(row["field"]) or pd.isna(row["academic_prerequisites"]):
        return 0
        
    prereq_text = str(row["academic_prerequisites"]).lower()
    field_text = str(row["field"]).lower()
    
    # Clean and split compound domain headers (e.g., "Computer Science / IT" -> ["computer science", "it"])
    fields_to_check = [f.strip() for f in re.split(r'[/|&,]', field_text) if f.strip() != ""]
    
    # Broad classification catch for generalist/uncategorised streams
    if any(unendorsed in fields_to_check for unendorsed in ["other", "short course / other", "arts / humanities"]):
        return 0
        
    # Check if any structural token substring matches the baseline prerequisites block
    for item in fields_to_check:
        if len(item) > 2 and item in prereq_text:
            return 1
        if item in ["it", "ai", "cs"] and re.search(r'\b' + re.escape(item) + r'\b', prereq_text):
            return 1
            
    return 0



In [192]:
def clean_university_catalog(raw_path, output_path):
    print(f"[INFO] Ingesting raw catalog entries from: {raw_path}")
    df = pd.read_csv(raw_path)
    
    # International eligibility
    df["is_intl_eligible"] = df["programme_eligibility_intl"].apply(lambda x: 0 if "NOT ELIGIBLE" in str(x).upper() or "DOMESTIC" in str(x).upper() else 1)
    
    # GPA / percentage
    df["min_std12_pct"] = df["min_std12_%"].apply(lambda x: extract_absolute_minimum_percentage(x, default=0.0))
    df["min_bachelor_pct"] = df["min_bachelor_requirement"].apply(parse_bachelor_percentage_regex)
    
    # English proficiency
    df["ielts_min_overall"] = df["ielts_overall"].apply(lambda x: extract_absolute_minimum_percentage(x, default=6.0))
    df["ielts_min_band"] = df["ielts_min_band"].apply(lambda x: extract_absolute_minimum_percentage(x, default=5.5))
    df["pte_min_overall"] = df["pte"].apply(lambda x: extract_absolute_minimum_percentage(x, default=50.0))
    df["toefl_min_overall"] = df["toefl"].apply(lambda x: extract_absolute_minimum_percentage(x, default=80.0))
    
    combined_context = df["academic_prerequisites"].fillna("") + " " + df["entrance_requirements"].fillna("") + " " + df["eligibility_note"].fillna("")
    df["min_bachelor_yrs"] = combined_context.apply(extract_bachelor_years_regex)
    df["substitute_bachelor_with_work_exp"] = combined_context.apply(extract_work_exp_substitution_regex)
    
    #Core-subject 
    df["field_prereq_match"] = df.apply(dynamically_match_field_prereq, axis=1)

    # Work experience parsing
    work_params = df["work_experience_required"].apply(parse_work_experience)
    df["work_exp_mandatory"] = [p[0] for p in work_params]
    df["min_work_exp_months"] = [p[1] for p in work_params]

    df["programme_clean"] = df["programme"].astype(str).str.strip()
    df["university_clean"] = df["university"].astype(str).str.strip()
    df.columns = df.columns.str.replace(r"\s+", "_", regex=True)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"[SUCCESS] Exported cleaned catalog matrix to: {output_path} ({len(df)} courses)")

In [193]:
clean_university_catalog("../data/raw/University_database.csv", "../data/processed/University_database_clean.csv")
clean_univ_df = pd.read_csv('../data/processed/University_database_clean.csv')
#clean_univ_df

[INFO] Ingesting raw catalog entries from: ../data/raw/University_database.csv
[SUCCESS] Exported cleaned catalog matrix to: ../data/processed/University_database_clean.csv (1567 courses)


### Student Data Cleanup

In [194]:
def standardize_percentage(val):
    """
    Standardizes decimal markers (e.g., 0.872 -> 87.2) into a standard 0-100% scale.
    Handles string parsing anomalies and filters out systemic invalid inputs.
    """
    if pd.isna(val):
        return np.nan
    try:
        # Cast value to float
        num = float(val)
        # If expressed as a decimal coefficient < 1.0 (e.g. 0.941), upscale to percent
        if 0.0 < num <= 1.0:
            return round(num * 100, 2)
        elif num > 1.0 <= 100.0:
            return round(num, 2)
        return np.nan
    except (ValueError, TypeError):
        return np.nan

In [195]:
def parse_backlogs(val):
    """
    Defensive parser to protect backlog fields against floating-point bleed.
    Ensures text decimals are handled safely without inflating values.
    """
    if pd.isna(val):
        return 0

    # Convert to string and strip whitespace
    val_str = str(val).strip()

    # CRITICAL: If a float leaked in, drop the fractional decimals completely first!
    if "." in val_str:
        val_str = val_str.split(".")[0]

    # Clean any remaining textual debris
    cleaned = re.sub(r"[^\d]", "", val_str)

    if cleaned == "":
        return 0

    return int(cleaned)

In [196]:
def derive_bachelor_duration_from_edu_level(edu_text):
    """
    Dynamically maps baseline background durations using structural indicators:
    Bachelor -> 3 Years, Master -> 5 Years, Doctorate -> 6 Years, else -> 0.
    """
    if pd.isna(edu_text):
        return 0
    clean_text = str(edu_text).strip().upper()
    
    if "BACHELOR" in clean_text:
        return 3
    elif "MASTER" in clean_text:
        return 5
    elif "DOCTOR" in clean_text or "PHD" in clean_text:
        return 6
    return 0

In [217]:
def clean_student_data(raw_path, output_path):
    print(f"[INFO] Ingesting raw student data from: {raw_path}")
    df = pd.read_excel(raw_path, sheet_name=0)
    df["12th_marks_clean"] = df["12th marks"].apply(standardize_percentage)
    df["bachelors_marks_clean"] = df["bachelors marks"].apply(standardize_percentage)
    df["number_of backlogs_clean"] = df["number of backlogs"].apply(parse_backlogs)
    df["bachelors_duration_years"] = df["ZKUT_HighestEdu (Student ID)"].apply(derive_bachelor_duration_from_edu_level)
    
    df["Test_type"] = df["Test type"].astype(str).str.strip().str.upper()
    df["Test_type"] = df["Test_type"].replace({
        "IELTS ACADEMIC": "IELTS", "PTE ACADEMIC": "PTE", 
        "TOEFL IBT": "TOEFL", "NAN": "NONE", "NOT ASSIGNED": "NONE"
    })
    df["Total_Score"] = pd.to_numeric(df["Total Score"], errors="coerce").fillna(0.0)

    test_scores = (
        df.loc[df["Test_type"].ne("NONE")]
        .pivot_table(
            index="Student ID",
            columns="Test_type",
            values="Total_Score",
            aggfunc="first"
        )
        .rename(columns=lambda test_type: f"{test_type}_Total_Score")
        .reset_index()
    )
    student_columns = [
        column for column in df.columns
        if column not in {"Test_type", "Total_Score", "Test type", "Total Score"}
    ]
    df = (
        df[student_columns]
        .drop_duplicates(subset="Student ID")
        .merge(test_scores, on="Student ID", how="left", validate="one_to_one")
    )    
    

    #df = df.dropna(subset=["12th marks_clean", "bachelors marks_clean"], how="any")
    
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.columns = df.columns.str.replace(r"\s+", "_", regex=True)
    df.to_csv(output_path, index=False)
    print(f"[SUCCESS] Exported cleaned students file to: {output_path} ({len(df)} rows)")


In [218]:
#stu_df = pd.read_excel('../data/raw/Student_Data_NZ_analysis.xlsx', sheet_name=0)
clean_student_data("../data/raw/Student_Data_NZ_analysis.xlsx", "../data/processed/Student_Data_NZ_analysis_clean.csv")
clean_student_df= pd.read_csv('../data/processed/Student_Data_NZ_analysis_clean.csv')
#clean_student_df

[INFO] Ingesting raw student data from: ../data/raw/Student_Data_NZ_analysis.xlsx
[SUCCESS] Exported cleaned students file to: ../data/processed/Student_Data_NZ_analysis_clean.csv (5465 rows)


### Rule Engine
- `PASS`: Student clears every program expectation floor directly.
- `CONDITIONAL`: Student falls slightly short on English parameters, degree duration, or core subjects, but qualifies for pre-sessional entry pathways or graduate diploma tracks.
- `FAIL`: Critical baseline metric gaps are present (e.g., academic metrics are far below limits or core subjects are missing), making them completely ineligible

In [211]:
class InstitutionalAdmissionsRuleEngine:
    """
    Implements a multi-tiered rule engine evaluating entries across 
    Academic background, Core-subjects, GPA/percentages, and English milestones.
    """
    def __init__(self, catalog_clean_path="../data/processed/University_database_clean.csv"):
        if not os.path.exists(catalog_clean_path):
            raise FileNotFoundError(f"[ERROR] Clean catalog file missing at: {catalog_clean_path}")
        self.catalog_df = pd.read_csv(catalog_clean_path)

    def evaluate_course_eligibility(self, student, course):
        """
        Evaluates a single student against a single course using the visual matrix criteria rules.
        """
        # --- RULE MATRIX GATE 1: INTERNATIONAL ENTRY ELIGIBILITY ---
        if int(course["is_intl_eligible"]) == 0:
            return "FAIL", "Program not open to international applications"
        
        # --- RULE MATRIX GATE 2: MASTERS-ONLY ENFORCEMENT FILTER ---
        # Checks the course level parameters and rejects Certificates, Diplomas, and Bachelors
        course_degree_type = str(course.get("degree_type", "")).strip().upper()
        course_level_type = str(course.get("level", "")).strip().upper()
        if "MASTER" not in course_degree_type or "MASTER" not in course_level_type:
            return "FAIL", f"Excluded: Exclusively recommending Master's tracks (encountered: {course['degree_type']})"

        decisions = {}

        # --- CRITERIA 1: ACADEMIC BACKGROUND (DEGREE DURATION / RECOGNITION) ---
        student_yrs = int(student.get("bachelors_duration_years", 3))
        required_yrs = int(course["min_bachelor_yrs"])
        
        if student_yrs >= required_yrs:
            decisions["academic"] = "PASS"
        elif int(course["substitute_bachelor_with_work_exp"]) == 1 and int(student.get("work_experience_months", 0)) >= 24:
            decisions["academic"] = "PASS" # Work experience bridges the gap cleanly
        else:
            # 3-year degree vs 4-year degree constraint issue detected
            decisions["academic"] = "CONDITIONAL" 

        # --- CRITERIA 2: CORE-SUBJECT RELEVANCE ---
        if int(course["field_prereq_match"]) == 1:
            decisions["core_subject"] = "PASS"
        else:
            decisions["core_subject"] = "CONDITIONAL"

        # --- CRITERIA 3: GPA / PERCENTAGE FLOOR ---
        student_ug = float(student.get("marks_bachelors", 0.0))
        target_ug = float(course["min_bachelor_pct"])
        
        if student_ug >= target_ug:
            decisions["gpa"] = "PASS"
        elif (target_ug - student_ug) <= 5.0:
            decisions["gpa"] = "CONDITIONAL"
        else:
            return "FAIL", "GPA/Percentage is far below minimum catalog thresholds"

        # --- CRITERIA 4: MULTI-EXAM ENGLISH PROFICIENCY GATE ---
        # The engine extracts all score vectors and checks if ANY test clears the requirements
        student_ielts = float(student.get("ielts_score", 0.0))
        student_pte = float(student.get("pte_score", 0.0))
        student_toefl = float(student.get("toefl_score", 0.0))
        student_min_band = float(student.get("ielts_min_band", 0.0))

        # Check explicit individual statuses for each test separately
        cleared_ielts = False
        cleared_pte = False
        cleared_toefl = False
        
        conditional_ielts = False

        # 1. Evaluate IELTS parameters if values are supplied
        if student_ielts > 0.0 and course["ielts_min_overall"] > 0.0:
            if student_ielts >= float(course["ielts_min_overall"]):
                # If an explicit individual sub-band test is passed, double-check constraints
                if student_min_band > 0.0 and course["ielts_min_band"] > 0.0:
                    if student_min_band >= float(course["ielts_min_band"]):
                        cleared_ielts = True
                else:
                    cleared_ielts = True
            elif (float(course["ielts_min_overall"]) - student_ielts) <= 0.5:
                conditional_ielts = True

        # 2. Evaluate PTE parameters if values are supplied
        if student_pte > 0.0 and course["pte_min_overall"] > 0.0:
            if student_pte >= float(course["pte_min_overall"]):
                cleared_pte = True

        # 3. Evaluate TOEFL parameters if values are supplied
        if student_toefl > 0.0 and course["toefl_min_overall"] > 0.0:
            if student_toefl >= float(course["toefl_min_overall"]):
                cleared_toefl = True

        # --- RESOLVE THE ENGLISH DISPOSITION DECISION GATES ---
        # If the course profile sets no floor requirements, auto-pass the milestone
        if course["ielts_min_overall"] == 0.0 and course["pte_min_overall"] == 0.0 and course["toefl_min_overall"] == 0.0:
            decisions["english"] = "PASS"
        # If student clears ANY of the target score thresholds, mark column as a clear PASS
        elif cleared_ielts or cleared_pte or cleared_toefl:
            decisions["english"] = "PASS"
        # If the student missed a direct pass but falls within range for a pre-sessional pathway
        elif conditional_ielts or (student_pte > 0.0) or (student_toefl > 0.0):
            decisions["english"] = "CONDITIONAL"
        else:
            return "FAIL", "Language marks do not meet direct or conditional minimum criteria thresholds"

        # --- COMBINE ALL TIERS TO GENERATE THE FINAL SYSTEM DECISION ---
        statuses = list(decisions.values())
        
        if "FAIL" in statuses:
            return "FAIL", "Rejected during criteria evaluation checkpoints"
        elif "CONDITIONAL" in statuses:
            return "CONDITIONAL", "Eligible via structural pathway tracking programs"
        else:
            return "PASS", "Direct admission criteria cleared successfully"

    def process_all_recommendations(self, student_profile):
        """
        Evaluates the student profile against every course in the catalog.
        Categorizes results into distinct operational dataframes.
        """
        passed_tracks = []
        conditional_tracks = []
        
        for _, course in self.catalog_df.iterrows():
            decision, reason = self.evaluate_course_eligibility(student_profile, course)
            course_summary = {
                "university": course["university_clean"],
                "programme": course["programme_clean"],
                "field": course["field"],
                "reasoning_notes": reason
            }
            
            if decision == "PASS":
                passed_tracks.append(course_summary)
            elif decision == "CONDITIONAL":
                conditional_tracks.append(course_summary)
                
        return pd.DataFrame(passed_tracks), pd.DataFrame(conditional_tracks)

In [213]:
engine = InstitutionalAdmissionsRuleEngine()

In [214]:
# Simulating a student who missed the IELTS target (scored 6.0 instead of 6.5) 
# but cleared the English requirement by providing a strong alternate PTE score
sample_student = {
    "marks_12th": 82.0,
    "marks_bachelors": 65.0,
    "bachelors_duration_years": 3,
    "ielts_score": 6.0,
    "ielts_min_band": 5.5,
    "pte_score": 61.0,           # This will trigger a PASS if the course requires PTE 58
    "toefl_score": 0.0,
    "work_experience_months": 12
}

direct_df, conditional_df = engine.process_all_recommendations(sample_student)

print(f"\n--- MULTI-EXAM EVALUATION COMPLETE ---")
print(f"Direct PASS Track Recommendations Count: {len(direct_df)}")
print(f"Conditional PATHWAY Recommendations Count: {len(conditional_df)}")
if not direct_df.empty:
    print(f"\nSample Successful Direct Matches:\n{direct_df.head(3)}")


--- MULTI-EXAM EVALUATION COMPLETE ---
Direct PASS Track Recommendations Count: 134
Conditional PATHWAY Recommendations Count: 339

Sample Successful Direct Matches:
           university                                     programme  \
0  Lincoln University                     Master of Applied Science   
1  Lincoln University                  Master of Tourism Management   
2  Lincoln University  Master of Applied Science - Land and Society   

                   field                                 reasoning_notes  
0                Science  Direct admission criteria cleared successfully  
1  Hospitality / Tourism  Direct admission criteria cleared successfully  
2                Science  Direct admission criteria cleared successfully  


In [224]:
# 1. Load your cleaned, standardized student DataFrame
students_clean_df = pd.read_csv("../data/processed/Student_Data_NZ_analysis_clean.csv")
raw_row = students_clean_df.iloc[2]
# Automatically construct the student input vector format

example_student = {
        "student_id": int(raw_row["Student_ID"]),
        "marks_12th": float(raw_row["12th_marks_clean"]),
        "marks_bachelors": float(raw_row["bachelors_marks_clean"]),
        "bachelors_duration_years": int(raw_row["bachelors_duration_years"]), # Dynamic value from ZKUT column
        "ielts_score": float(raw_row["IELTS_Total_Score"]),
        "pte_score": float(raw_row["PTE_Total_Score"]),
        "toefl_score": float(raw_row["TOEFL_Total_Score"]),
        "work_experience_months": 0 
    }

engine = InstitutionalAdmissionsRuleEngine()
direct_df, conditional_df = engine.process_all_recommendations(example_student)

print(f"\n--- SYSTEMS PROCESSING REPORT FOR STUDENT: {example_student['student_id']} ---")
print(f"Calculated Prior Bachelor's Duration Vector: {example_student['bachelors_duration_years']} Years.")
print(f"Calculated marks_12th Vector: {example_student['marks_12th']} Points.")
print(f"Calculated marks_bachelors Vector: {example_student['marks_bachelors']} Points.")
print(f"Calculated ielts_score: {example_student['ielts_score']}, pte_score: {example_student['pte_score']}, toefl_score: {example_student['toefl_score']}")
print(f"Directly Eligible Masters Tracks Discovered: {len(direct_df)}")
print(f"Surfaced Pathway Masters Options: {len(conditional_df)}")
if not direct_df.empty:
    print(f"\nDirect Pass Snippet:\n{direct_df[['university', 'programme', 'degree_type']].head(2)}")


--- SYSTEMS PROCESSING REPORT FOR STUDENT: 67474225 ---
Calculated Prior Bachelor's Duration Vector: 0 Years.
Calculated marks_12th Vector: 94.7 Points.
Calculated marks_bachelors Vector: 71.7 Points.
Calculated ielts_score: 6.5, pte_score: nan, toefl_score: nan
Directly Eligible Masters Tracks Discovered: 0
Surfaced Pathway Masters Options: 473


### Content Filtering - The Personalization Ranking Layer
standard machine learning approach: TF-IDF (Term Frequency-Inverse Document Frequency) Vectorization and Cosine Similarity.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class CourseContentSimilarityRanker:
    """
    Stage 2A Machine Learning Layer: Uses TF-IDF and Cosine Similarity 
    to calculate text-affinity scores between a student's prior field 
    and the text profiles of your Master's course catalog.
    """
    def __init__(self, processed_catalog_path="../data/processed/University_database_clean.csv"):
        if not os.path.exists(processed_catalog_path):
            raise FileNotFoundError(f"[ERROR] Processed university catalog file missing at: {processed_catalog_path}")
        
        # Load the catalog dataset
        self.catalog_df = pd.read_csv(processed_catalog_path)
        
        # Enforce your strict structural rule: Keep ONLY Master's programs
        self.masters_catalog = self.catalog_df[
            self.catalog_df["degree_type"].astype(str).str.upper().str.contains("MASTER")
        ].copy().reset_index(drop=True)
        
        # Initialize the Machine Learning text processing assets
        self.vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
        self.tfidf_matrix = None
        
        # Run internal setup
        self.
        ()

    def _build_catalog_text_embeddings(self):
        """
        Concatenates critical descriptive text columns for each Master's program 
        and computes the baseline TF-IDF training matrix map.
        """
        # Combine distinct descriptive columns to build a rich textual fingerprint
        combined_text_metadata = (
            self.masters_catalog["programme_clean"].fillna("") + " " +
            self.masters_catalog["field"].fillna("") + " " +
            self.masters_catalog["typical_roles"].fillna("") + " " +
            self.masters_catalog["field"].fillna("") # Repeated to apply intentional weight
        )
        
        # Fit the TF-IDF model and transform the catalog strings into numeric vectors
        self.tfidf_matrix = self.vectorizer.fit_transform(combined_text_metadata)
        print(f"[INFO] Content Filter vector space initialized across {self.masters_catalog.shape[0]} Master's courses.")

    def calculate_course_affinity_scores(self, student_background_text, eligible_candidates_df):
        """
        Computes the real-time Cosine Similarity score between a student's text profile 
        and the filtered candidate courses passed out of Stage 1.
        """
        if eligible_candidates_df.empty:
            print("[WARNING] Passed candidate pool is completely empty. Skipping similarity scores.")
            return eligible_candidates_df

        # 1. Transform the student's background text into the same vector space
        student_vector = self.vectorizer.transform([str(student_background_text).lower()])
        
        # 2. Extract the matching subset from our pre-trained catalog matrix
        # Maps the filtered candidates array securely back onto our vector indices
        eligible_programme_names = eligible_candidates_df["programme_clean"].tolist()
        
        scored_rows = []
        for _, row in eligible_candidates_df.iterrows():
            prog_name = row["programme_clean"]
            
            # Locate the exact row coordinates inside our master's catalog array
            matched_indices = self.masters_catalog.index[self.masters_catalog["programme_clean"] == prog_name].tolist()
            
            if matched_indices:
                catalog_idx = matched_indices[0]
                course_vector = self.tfidf_matrix[catalog_idx]
                
                # Compute the directional angle (similarity score) between user and course vectors
                similarity_score = cosine_similarity(student_vector, course_vector)[0][0]
                
                # Scale from 0.0-1.0 to a clean 0-100 score format
                row["affinity_match_score"] = round(float(similarity_score) * 100, 2)
            else:
                row["affinity_match_score"] = 0.0
                
            scored_rows.append(row)
            
        output_df = pd.DataFrame(scored_rows)
        # Sort the output matrix descending, prioritizing highest textual affinity
        return output_df.sort_values(by="affinity_match_score", ascending=False).reset_index(drop=True)

In [236]:
# Mock runtime simulation validating vector pipeline performance
ranker = CourseContentSimilarityRanker(processed_catalog_path="../data/processed/University_database_clean.csv")

# Simulating a subset pool that passed the Stage 1 rule engine filter block
# (Using a slice of your dataset rows containing only master's programs for test accuracy)
mock_stage1_output = ranker.masters_catalog.head(5).copy()

# Inputting a sample student textual background descriptor text
mock_student_background = "Computer Science and IT data analysis software engineering roles"

scored_results = ranker.calculate_course_affinity_scores(mock_student_background, mock_stage1_output)

print(f"\n--- MACHINE LEARNING CONTENT RANKING VISUALIZATION ---")
print(f"Input Student Prior Focus Domain: '{mock_student_background}'")
print(scored_results[["university_clean", "programme_clean", "affinity_match_score"]])

[INFO] Content Filter vector space initialized across 495 Master's courses.

--- MACHINE LEARNING CONTENT RANKING VISUALIZATION ---
Input Student Prior Focus Domain: 'Computer Science and IT data analysis software engineering roles'
           university_clean                         programme_clean  \
0    University of Auckland           Master of Engineering Studies   
1        Lincoln University               Master of Applied Science   
2        Lincoln University  Master of Water Science and Management   
3  University of Canterbury                          Master of Arts   
4        Lincoln University                 Master of Bioprotection   

   affinity_match_score  
0                 18.46  
1                  7.74  
2                  1.89  
3                  0.00  
4                  0.00  
